# Notebook 04 — Camada Gold (Modelagem Dimensional)

## Star Schema (Esquema Estrela)

O modelo dimensional Star Schema organiza os dados em:
- **Tabelas de Dimensão (Dim)**: atributos descritivos que respondem "quem, o quê, onde, quando"
- **Tabela de Fatos (Fact)**: métricas numéricas e chaves estrangeiras para as dimensões

### Diagrama do Star Schema

```
                    dim_clientes
                    (id_cliente)
                         |
     dim_produtos       |        dim_calendario
     (id_produto)       |        (data)
          \             |             /
           \            |            /
            \           |           /
             \          |          /
              \         |         /
               \        |        /
            fact_vendas (fatos)
                    |
             dim_vendedores
             (id_vendedor)
```

Neste notebook criamos todas as tabelas dimensionais e a tabela fato.


## 1. Criação da SparkSession


In [ ]:
import os
from datetime import date, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, date_add, dayofmonth, month, year, quarter, date_format, monotonically_increasing_id, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DateType, StructType, StructField, StringType

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
SILVER_DIR = os.path.join(DATA_DIR, "silver")
GOLD_DIR = os.path.join(DATA_DIR, "gold")

spark = (
    SparkSession.builder
    .appName("NB04_Gold_Modelagem")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession iniciada. Versão: {spark.version}")


## 2. dim_clientes

Dimensão de clientes com atributos descritivos: `id_cliente`, `nome`, `cidade`, `estado`, `regiao`.
Lida da camada Silver, remove duplicatas por `id_cliente`.


In [ ]:
df_silver_clientes = spark.read.format("delta").load(os.path.join(SILVER_DIR, "clientes"))

df_dim_clientes = df_silver_clientes \
    .select(
        col("cliente_id").alias("id_cliente"),
        col("nome"),
        col("cidade"),
        col("estado"),
        col("regiao")
    ) \
    .dropDuplicates(["id_cliente"])

dim_clientes_path = os.path.join(GOLD_DIR, "dim_clientes")
df_dim_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(dim_clientes_path)

print(f"[OK] dim_clientes: {df_dim_clientes.count()} registros")
df_dim_clientes.show(5, truncate=False)


## 3. dim_produtos

Dimensão de produtos: `id_produto`, `nome_produto`, `categoria`, `preco_venda`.


In [ ]:
df_silver_produtos = spark.read.format("delta").load(os.path.join(SILVER_DIR, "produtos"))

df_dim_produtos = df_silver_produtos \
    .select(
        col("sku").alias("id_produto"),
        col("nome_produto"),
        col("categoria"),
        col("preco_venda")
    ) \
    .dropDuplicates(["id_produto"])

dim_produtos_path = os.path.join(GOLD_DIR, "dim_produtos")
df_dim_produtos.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(dim_produtos_path)

print(f"[OK] dim_produtos: {df_dim_produtos.count()} registros")
df_dim_produtos.show(5, truncate=False)


## 4. dim_calendario

Dimensão de datas com colunas derivadas: `data`, `dia`, `mes`, `nome_mes`, `mes_num`, `trimestre`, `ano`.
Os nomes dos meses estão em português. O intervalo cobre o período dos pedidos (2023–2024).


In [ ]:
meses_pt = [
    "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
]

df_silver_pedidos = spark.read.format("delta").load(os.path.join(SILVER_DIR, "pedidos"))

from pyspark.sql.functions import min as spark_min, max as spark_max
min_date = df_silver_pedidos.select(spark_min("data_pedido")).collect()[0][0]
max_date = df_silver_pedidos.select(spark_max("data_pedido")).collect()[0][0]
print(f"Período dos pedidos: {min_date} a {max_date}")

num_dias = (max_date - min_date).days + 1
df_calendario = spark.range(num_dias) \
    .select(
        date_add(lit(min_date), col("id").cast(IntegerType())).alias("data")
    ) \
    .withColumn("dia", dayofmonth(col("data"))) \
    .withColumn("mes_num", month(col("data"))) \
    .withColumn("ano", year(col("data"))) \
    .withColumn("trimestre", quarter(col("data")))

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def nome_mes_pt(mes_num):
    if mes_num is None:
        return None
    return meses_pt[int(mes_num) - 1]

df_calendario = df_calendario \
    .withColumn("nome_mes", nome_mes_pt(col("mes_num"))) \
    .withColumn("mes", col("mes_num"))

df_dim_calendario = df_calendario.select("data", "dia", "mes", "nome_mes", "mes_num", "trimestre", "ano")

dim_calendario_path = os.path.join(GOLD_DIR, "dim_calendario")
df_dim_calendario.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(dim_calendario_path)

print(f"[OK] dim_calendario: {df_dim_calendario.count()} registros (de {min_date} a {max_date})")
df_dim_calendario.show(12, truncate=False)


## 5. dim_vendedores

Dimensão com 15 vendedores artificiais. Cada vendedor possui `id_vendedor`, `nome` e `regiao`.


In [ ]:
vendedores_data = [
    (1, "Roberto Alves", "Sudeste"),
    (2, "Carla Mendes", "Sudeste"),
    (3, "Márcio Teixeira", "Nordeste"),
    (4, "Patrícia Lopes", "Sul"),
    (5, "José Barros", "Nordeste"),
    (6, "Sandra Matos", "Sudeste"),
    (7, "Antônio Rangel", "Centro-Oeste"),
    (8, "Denise Quadros", "Norte"),
    (9, "Jorge Fonseca", "Sul"),
    (10, "Lúcia Vianna", "Sudeste"),
    (11, "Wilson Borges", "Nordeste"),
    (12, "Márcia Leal", "Centro-Oeste"),
    (13, "Hélio Prado", "Norte"),
    (14, "Vera Coelho", "Sul"),
    (15, "Nelson Guará", "Sudeste"),
]

df_dim_vendedores = spark.createDataFrame(vendedores_data, ["id_vendedor", "nome", "regiao"])

dim_vendedores_path = os.path.join(GOLD_DIR, "dim_vendedores")
df_dim_vendedores.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(dim_vendedores_path)

print(f"[OK] dim_vendedores: {df_dim_vendedores.count()} registros")
df_dim_vendedores.show(15, truncate=False)


## 6. fact_vendas

Tabela fato central com as métricas de vendas e chaves estrangeiras para todas as dimensões.
Os dados vêm da `silver/pedidos` enriquecida.


In [ ]:
df_silver_pedidos_full = spark.read.format("delta").load(os.path.join(SILVER_DIR, "pedidos"))

df_fact_vendas = df_silver_pedidos_full \
    .select(
        col("pedido_id"),
        col("cliente_id").alias("id_cliente"),
        col("sku").alias("id_produto"),
        col("data_pedido"),
        col("quantidade"),
        col("valor_frete"),
        col("total_pedido"),
        col("municipio"),
        col("estado"),
        col("regiao"),
        col("id_vendedor")
    )

fact_vendas_path = os.path.join(GOLD_DIR, "fact_vendas")
df_fact_vendas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(fact_vendas_path)

print(f"[OK] fact_vendas: {df_fact_vendas.count()} registros")
df_fact_vendas.show(10, truncate=False)


## 7. Resumo da Camada Gold

Contagem final de todas as tabelas do Star Schema.


In [ ]:
print("=" * 60)
print("CAMADA GOLD — STAR SCHEMA")
print("=" * 60)

tabelas = [
    ("dim_clientes", dim_clientes_path),
    ("dim_produtos", dim_produtos_path),
    ("dim_calendario", dim_calendario_path),
    ("dim_vendedores", dim_vendedores_path),
    ("fact_vendas", fact_vendas_path),
]

for nome, path in tabelas:
    df = spark.read.format("delta").load(path)
    count = df.count()
    print(f"  {nome:<20} {count:>6} registros")

print("-" * 60)
total_dim = sum(spark.read.format("delta").load(p).count() for n, p in tabelas if n != "fact_vendas")
print(f"  {'Total Dimensões':<20} {total_dim:>6} registros (4 tabelas)")


## 8. Encerramento da SparkSession


In [ ]:
spark.stop()
print("SparkSession encerrada.")


## Conclusão

O Star Schema foi criado com sucesso na camada Gold:

- **dim_clientes**: 1.500 clientes com dados demográficos
- **dim_produtos**: 250 produtos com categorias e preços
- **dim_calendario**: ~700 dias cobrindo 2023–2024 com atributos temporais completos em português
- **dim_vendedores**: 15 vendedores com nome e região
- **fact_vendas**: 8.000 fatos transacionais com chaves para todas as dimensões

No próximo notebook (**NB05**), faremos o enriquecimento com agregações analíticas.
